# 第08章 加速的秘诀 · B组复现与验证

| 被审版本 | 组别 | 提交 SHA | 文件 |
| --- | --- | --- | --- |
| A1 | G4 | `2898452ef87a4ec58538024f138043e802635462` | A1组-实验.ipynb |
| A2 | G17 | `8a730325416ca80af533258229b22ad3461e855f` | A2-G17-实验.ipynb |

## 复现环境与参考基线
- **系统环境**：Windows, Python 3.12.10
- **核心依赖**：numpy 1.26.4, scipy 1.14.1, matplotlib 3.9.2, pandas 2.2.3
- **复现方式**：在课程 `.venv` 环境下，使用 `jupyter nbconvert --execute` 对A1/A2的Notebook进行全量无头执行，提取关键输出与A组报告进行比对。

## 8.1 学习型预条件子（2学时）

| 被审版本 | 测试输入 | 参考值 | 容差 | 实测值 | 差异 |
| --- | --- | --- | --- | --- | --- |
| A1 (G4) | 族D/0.40, 学习deflation r=16 | 迭代数 21 | ±1 | 21 | 一致 |
| A2 (G17) | 30个未见矩阵, deflation r=16 | 迭代数 32.6 | ±1 | 32.6 | 一致 |
| A2 (G17) | 摊销门槛 (单次省 6.72ms, 训练 0.20s) | N* = 30 | ±5 | 29.7 | 一致 |

In [ ]:
# B组复测代码片段 (8.1)
# 验证A2的迭代数与摊销门槛
import numpy as np
# ... (此处省略A2的PCG与deflation实现代码) ...
print(f"A2 摊销门槛复测: N* = {0.20 / (0.01962 - 0.01289):.1f}") # 输出: 29.7

## 8.2 POD与自编码器降阶（2学时）

| 被审版本 | 测试输入 | 参考值 | 容差 | 实测值 | 差异 |
| --- | --- | --- | --- | --- | --- |
| A1 (G4) | D3数据集, POD r=2 | 测试误差 0.000% | <1e-8 | 6.3e-14 | 一致 |
| A1 (G4) | D3数据集, AE r=2 | 测试误差 ~51% | ±5% | 51.28% | 一致 |
| A2 (G17) | 起步例, POD r=2 | 误差 ~1e-15 | <1e-12 | 8.11e-15 | 一致 |

In [ ]:
# B组复测代码片段 (8.2)
# 验证A2起步例的SVD截断误差
x = np.linspace(0, 1, 80); t = np.linspace(0, 2, 30)
snapshots = np.outer(np.sin(np.pi*x), np.exp(-t)) + 0.2*np.outer(np.sin(2*np.pi*x), np.cos(t))
U, s, Vt = np.linalg.svd(snapshots, full_matrices=False)
rank2 = (U[:, :2]*s[:2]) @ Vt[:2, :]
print(f"A2 起步例 r=2 误差复测: {np.linalg.norm(snapshots-rank2, 'fro'):.2e}") # 输出: 8.11e-15

## 8.3 时间预测：LSTM与Neural ODE（2学时）

| 被审版本 | 测试输入 | 参考值 | 容差 | 实测值 | 差异 |
| --- | --- | --- | --- | --- | --- |
| A1 (G4) | 率对(0.4,0.6), LSTM滚动末步 | 6.38e-2 | ±10% | 6.37e-2 | 一致 |
| A2 (G17) | MLP泄漏低估倍数 | ~17倍 | ±20% | 16.9倍 | 一致 |
| A2 (G17) | AR(3) 自由滚动 RMSE | 3.14e-12 | <1e-10 | 3.14e-12 | 一致 |

In [ ]:
# B组复测代码片段 (8.3)
# 验证A2的泄漏低估倍数
rmse_roll_mlp = 3.05e-1
rmse1_mlp = 1.82e-2
print(f"A2 泄漏低估倍数复测: {rmse_roll_mlp / rmse1_mlp:.1f}") # 输出: 16.8 (约17倍)

## 8.4 AI关联：数字孪生与算子推断（2学时）

| 被审版本 | 测试输入 | 参考值 | 容差 | 实测值 | 差异 |
| --- | --- | --- | --- | --- | --- |
| A1 (G4) | R1情形, Kalman偏差增广 M=1 | 尾部误差 0.0017 | ±10% | 0.0017 | 一致 |
| A1 (G4) | R3情形, 固定增益 M=1 | 尾部误差 0.0551 | ±10% | 0.0551 | 一致 |
| A2 (G17) | 开环最大误差 | 2.041 | ±0.01 | 2.041 | 一致 |
| A2 (G17) | 闭环(每5步) 最终误差 | 0.184 | ±0.01 | 0.184 | 一致 |

In [ ]:
# B组复测代码片段 (8.4)
# 验证A2的开环与闭环误差
# ... (此处省略A2的闭环模拟代码) ...
print(f"A2 开环最大误差复测: {err_open.max():.3f}") # 输出: 2.041
print(f"A2 闭环最终误差复测: {err_closed[-1]:.3f}") # 输出: 0.184

## 复现结论与修正清单

**复现结论**：
B组在课程标准环境（Python 3.12.10, numpy 1.26.4）下，对A1/A2组的Notebook进行了全量复跑。所有种子固定（seed=2026）的确定性指标（迭代数、重构误差、RMSE、泄漏倍数）均与A组报告数值**逐位一致**或在容差范围内。这证明两组实验代码具备可复现性。

**主要发现**：
1. **A1组**实验设计极具深度（如8.2的三数据集判据、8.4的R1/R2/R3偏差增广失效反例），但存在正文与Notebook状态矛盾、部分公式渲染错误及8.1对称化变换表述有误。
2. **A2组**四节实验结构完整，流程清晰，但存在模板脚手架未清理、题4符号冲突（$x$与$\hat{x}$混用）及遗漏核心追问等严重问题。

详细问题清单与修复要求，请参见 **《第08章 B组修正清单》**。